In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [5]:
from sklearn.metrics import classification_report
ths = [19, 20, 21, 22, 23, 24, 25, 26]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 19 hidden = 0


,0,1,2,3,4,5,-1
0,0,151125,0,0,0,0,51103
1,0,125485,41,0,362,1590,1058
2,0,6,82225,0,0,0,74
3,0,0,3,60922,0,0,25
4,0,66,0,0,45485,168,71
5,0,2,0,0,6,133,6
-1,0,0,0,0,0,0,0


th: 19 hidden: 0 acc:0.7069771288339782 recall:0.2526999228593469 precision:0.9764220341249976 f1:0.40149274252155637
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.98      0.25      0.40    202228
           1       0.45      0.98      0.62    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      0.99      0.99     45790
           5       0.07      0.90      0.13       147

    accuracy                           0.70    519956
   macro avg       0.75      0.85      0.69    519956
weighted avg       0.85      0.70      0.67    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,149546,0,0,0,0,52682
1,0,125171,18,0,360,1540,1447
2,0,6,82036,0,0,0,263
3,0,0,3,60876,0,0,71
4,0,61,0,0,45431,168,130
5,0,2,0,0,6,130,9
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.7086945818492334 recall:0.2605079415313409 precision:0.964836452877184 f1:0.41024802398473703
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.96      0.26      0.41    202228
           1       0.46      0.97      0.62    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      0.99      0.99     45790
           5       0.07      0.88      0.13       147

    accuracy                           0.70    519956
   macro avg       0.75      0.85      0.69    519956
weighted avg       0.85      0.70      0.68    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 24 hidden: 5 acc:0.9570867534945264 recall:0.8707482993197279 precision:0.0057086789760057086 f1:0.01134299260046967
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.01      0.87      0.01       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.88      0.94    128536
           2       1.00      0.97      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.95      0.97     45790

    accuracy                           0.96    519956
   macro avg       0.83      0.94      0.82    519956
weighted avg       1.00      0.96      0.98    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,196598,0,0,0,0,0,5630
1,75,107807,3,1,22,0,20628
2,0,0,78886,0,0,0,3419
3,0,0,3,60806,0,0,141
4,1,4,0,0,42643,0,3142
5,0,4,0,0,6,0,137
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.9365907884513305 recall:0.9319727891156463 precision:0.0041393479771580505 f1:0.008242088797978582
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.93      0.01       147
           0       1.00      0.97      0.99    202228
           1       1.00      0.84      0.91    128536
           2       1.00      0.96      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.93      0.96     45790

    accuracy                           0.94    519956
   macro avg       0.83      0.94      0.81    519956
weighted avg       1.00      0.94      0.97    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,193804,0,0,0,0,0,8424
1,64,99535,3,0,5,0,28929
2,0,0,74178,0,0,0,8127
3,0,0,3,60730,0,0,217
4,0,0,0,0,38422,0,7368
5,0,0,0,0,1,0,146
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.8979413642692843 recall:0.9931972789115646 precision:0.002743793576516134 f1:0.005472468983095318
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.99      0.01       147
           0       1.00      0.96      0.98    202228
           1       1.00      0.77      0.87    128536
           2       1.00      0.90      0.95     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.84      0.91     45790

    accuracy                           0.90    519956
   macro avg       0.83      0.91      0.79    519956
weighted avg       1.00      0.90      0.94    519956



# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 19: 0.5496945330131294
Média f1 absolute th 20: 0.544319781600469
Média f1 absolute th 21: 0.5376514587124832
Média f1 absolute th 22: 0.5829313321014497
Média f1 absolute th 23: 0.6495463027363966
Média f1 absolute th 24: 0.623343010976607
Média f1 absolute th 25: 0.6040131104513442
Média f1 absolute th 26: 0.5695814155531184
